In [1]:
# Import
import requests
import geopandas as gpd
import math
import mapbox_vector_tile
from shapely.geometry import shape
import pandas as pd

# Query via Features API

In [2]:
# Define BGT API endpoint
base_url = "https://api.pdok.nl/lv/bgt/ogc/v1/collections/wegdeel/items"

# Define bounding box (Amsterdam as test)
# ams_box = "4.8,52.32,5.0,52.41"
ams_box = "4.937739,52.311523,4.946623,52.316654"

# Temporal filter
temporal_date = "2025-12-31T00:00:00Z" # When VHR acquisition is finished

# Define parameters for extraction
params = {
    "f": "json",
    "bbox": ams_box,
    "datetime": temporal_date
}

In [3]:
# Build function to query and filter parking data
def get_parking(base_url, params):
    # Initialize list to store result
    features = []
    page = 1
    # Request data
    while base_url:
        print(f"requesting page {page}")
        response = requests.get(base_url, params)
        # End code when error
        if response.status_code != 200:
            print(response.status_code)
            break
        
        data = response.json()
        features.extend(data.get("features", [])) # Extend features to initial list
        
        # Handle pagination
        base_url = next((l['href'] for l in data.get('links', []) if l['rel'] == 'next'), None)
        params = None
        page = page + 1
        
    # Convert to gdf
    gdf = gpd.GeoDataFrame.from_features(features)
    
    # Filter for parking lot
    park_lot = gdf[gdf["functie"] == "parkeervlak"]
    
    return park_lot

In [4]:
parking_space = get_parking(base_url, params=params)

requesting page 1
requesting page 2
requesting page 3
requesting page 4
requesting page 5
requesting page 6
requesting page 7
requesting page 8
requesting page 9
requesting page 10
requesting page 11
requesting page 12
requesting page 13
requesting page 14
requesting page 15
requesting page 16
requesting page 17
requesting page 18
requesting page 19
requesting page 20
requesting page 21
requesting page 22
requesting page 23
requesting page 24
requesting page 25
requesting page 26
requesting page 27
requesting page 28
requesting page 29
requesting page 30
requesting page 31
requesting page 32
requesting page 33
requesting page 34
requesting page 35
requesting page 36
requesting page 37
requesting page 38
requesting page 39
requesting page 40
requesting page 41
requesting page 42
requesting page 43
requesting page 44
requesting page 45
requesting page 46
requesting page 47
requesting page 48
requesting page 49
requesting page 50
requesting page 51
requesting page 52
requesting page 53
re

In [8]:
parking_space = parking_space.set_crs(4326)

In [9]:
parking_space.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [10]:
parking_space.to_file("../data/referenced_data/parking_bgt_ams.gpkg", driver="GPKG")

In [4]:
johan_cruijff = get_parking(base_url, params=params)

requesting page 1
requesting page 2
requesting page 3
requesting page 4
requesting page 5
requesting page 6
requesting page 7
requesting page 8
requesting page 9
requesting page 10
requesting page 11
requesting page 12
requesting page 13
requesting page 14
requesting page 15
requesting page 16
requesting page 17
requesting page 18
requesting page 19
requesting page 20
requesting page 21
requesting page 22
requesting page 23
requesting page 24
requesting page 25
requesting page 26
requesting page 27
requesting page 28


In [ ]:
johan_cruijff.set_crs(4326)

In [7]:
johan_cruijff.to_file("../data/referenced_data/jc.gpkg", driver="GPKG")

c:\Users\Dimas RD\miniconda3\envs\geo_env\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [12]:
parking_space.describe()

,relatieve_hoogteligging
count,27505.000000
mean,0.001963
std,0.044266
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


# Query via Tiles API

In [2]:
# Define the tile template for RD New Quad
tile_url = "https://api.pdok.nl/lv/bgt/ogc/v1/tiles/NetherlandsRDNewQuad/{z}/{y}/{x}?f=mvt"

def get_rd_indices(x, y, zoom):
    """Calculates tile column and row for EPSG:28992."""
    origin_x, origin_y = -285401.92, 903401.92
    res = 3440.64 / (2**zoom) 
    tile_size_m = res * 256
    col = math.floor((x - origin_x) / tile_size_m)
    row = math.floor((origin_y - y) / tile_size_m)
    return col, row

def download_bgt_parking_batch(bbox, zoom):
    min_x, min_y, max_x, max_y = bbox
    col_min, row_max = get_rd_indices(min_x, min_y, zoom)
    col_max, row_min = get_rd_indices(max_x, max_y, zoom)
    
    all_features = []
    print(f"Downloading grid: Cols {col_min}-{col_max}, Rows {row_min}-{row_max}")

    for col in range(col_min, col_max + 1):
        for row in range(row_min, row_max + 1):
            url = tile_url.format(z=zoom, x=col, y=row)
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    # Decode MVT
                    data = mapbox_vector_tile.decode(response.content)
                    # BGT tiles usually have 'wegdeel' as the primary layer
                    if 'wegdeel' in data:
                        for feat in data['wegdeel']['features']:
                            # Filter for parking function
                            if feat['properties'].get('plus_functie') == 'parkeervlak' or \
                               feat['properties'].get('functie') == 'parkeervlak':
                                
                                geom = shape(feat['geometry'])
                                all_features.append({
                                    'geometry': geom,
                                    **feat['properties']
                                })
            except Exception as e:
                print(f"Error at {col}/{row}: {e}")

    if not all_features:
        return None

    # Create GeoDataFrame in RD New
    gdf = gpd.GeoDataFrame(all_features, crs="EPSG:28992")
    return gdf

In [3]:
# Execute for Amsterdam
ams_bbox = (115938.7728, 482453.8627, 126683.3628, 490968.5698)
parking_gdf = download_bgt_parking_batch(ams_bbox, 14)

Error at 7542/7724: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Error at 7542/7725: HTTPSConnectionPool(host='api.pdok.nl', port=443): Max retries exceeded with url: /lv/bgt/ogc/v1/tiles/NetherlandsRDNewQuad/14/7725/7542?f=mvt (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002484A92AC10>: Failed to resolve 'api.pdok.nl' ([Errno 11001] getaddrinfo failed)"))
Error at 7542/7726: HTTPSConnectionPool(host='api.pdok.nl', port=443): Max retries exceeded with url: /lv/bgt/ogc/v1/tiles/NetherlandsRDNewQuad/14/7726/7542?f=mvt (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002484A92B9D0>: Failed to resolve 'api.pdok.nl' ([Errno 11001] getaddrinfo failed)"))
Error at 7542/7727: HTTPSConnectionPool(host='api.pdok.nl', port=443): Max retries exceeded with url: /lv/bgt/ogc/v1/tiles/NetherlandsRDNewQuad/14/7727/7542?f=mvt (Caused by NameR

In [6]:
print(parking_gdf)

None
